# cAIuldron - AI Recipe Generator Pipeline (Optimized)

**Complete pipeline: Photo → Ingredient Detection → Nutrition → Recipes**

## Features:
- 🔍 Multi-ingredient detection (CLIP + DETR)
- 🥗 Nutrition estimation (529 ingredients)
- 🍳 Multi-model recipe generation (GPT-2, Llama 3.2 1B, Llama 3.1 8B)
- 🌐 Beautiful Gradio web interface
- 💯 100% local, no API costs

## 1. Environment Setup

In [1]:
import os
import sys
import json
import time
import re
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

from transformers import CLIPProcessor, CLIPModel, DetrImageProcessor, DetrForObjectDetection
import torch

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✓ Packages imported successfully")
print(f"  - PyTorch version: {torch.__version__}")
print(f"  - CUDA available: {torch.cuda.is_available()}")

✓ Packages imported successfully
  - PyTorch version: 2.7.1+cu118
  - CUDA available: True


## 2. Configure Paths and Parameters

In [2]:
PROJECT_ROOT = Path.cwd().parent.parent
MODEL_DIR = PROJECT_ROOT / "models" / "recipe_generation"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = DATA_DIR / "results" / "pipeline_output"
TEST_IMAGES_DIR = DATA_DIR / "test_images"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Detection settings
INGREDIENTS_CSV = DATA_DIR / "ingredients_vocabulary.csv"
DETECTION_MODE = "multi"  # "single" or "multi"
INGREDIENT_CONFIDENCE_THRESHOLD = 0.15
OBJECT_DETECTION_THRESHOLD = 0.7

# Recipe generation settings
NUM_RECIPES = 5

print("✓ Configuration loaded")
print(f"  - Project root: {PROJECT_ROOT}")
print(f"  - Detection mode: {DETECTION_MODE}")
print(f"  - Number of recipes: {NUM_RECIPES}")

✓ Configuration loaded
  - Project root: c:\Users\Champion\Documents\GitHub\cAIuldron
  - Detection mode: multi
  - Number of recipes: 5


## 3. Load CLIP and DETR Models

In [3]:
if not INGREDIENTS_CSV.exists():
    raise FileNotFoundError(f"Ingredients CSV not found: {INGREDIENTS_CSV}")

df = pd.read_csv(INGREDIENTS_CSV)
INGREDIENT_CANDIDATES = df['Ingredient'].tolist()

print(f"✓ Loaded {len(INGREDIENT_CANDIDATES)} ingredients")

# Load CLIP model
try:
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    print("✓ CLIP model loaded")
except Exception as e:
    print(f"✗ Failed to load CLIP model: {e}")
    clip_model = None
    clip_processor = None

# Load DETR for multi-ingredient detection
if DETECTION_MODE == "multi":
    try:
        detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
        detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
        print("✓ DETR model loaded (multi-ingredient detection)")
    except Exception as e:
        print(f"✗ Failed to load DETR: {e}")
        detr_model = None
        detr_processor = None
else:
    detr_model = None
    detr_processor = None

✓ Loaded 528 ingredients
✓ CLIP model loaded


Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✓ DETR model loaded (multi-ingredient detection)


## 4. Multi-Model System for Recipe Generation

In [4]:
from enum import Enum
from dataclasses import dataclass
from typing import Dict, Any

class RecipeModelType(Enum):
    """Available recipe generation models"""
    GPT2 = "gpt2"
    LLAMA_1B = "llama3.2-1b"
    LLAMA_8B_GGUF = "llama3.1-8b-gguf"

@dataclass
class ModelConfig:
    name: str
    display_name: str
    speed: str
    quality: str
    vram_required: str
    description: str

MODEL_INFO = {
    RecipeModelType.GPT2: ModelConfig(
        name="gpt2-finetuned",
        display_name="GPT-2 (Fast)",
        speed="fast",
        quality="good (60-70/100)",
        vram_required="1-2 GB",
        description="Fast but lower quality"
    ),
    RecipeModelType.LLAMA_1B: ModelConfig(
        name="llama3.2-1b-finetuned",
        display_name="Llama 3.2 1B (Recommended)",
        speed="medium",
        quality="excellent (90-95/100)",
        vram_required="4-5 GB",
        description="Recommended: Fast, high quality"
    ),
    RecipeModelType.LLAMA_8B_GGUF: ModelConfig(
        name="llama3.1-8b-gguf",
        display_name="Llama 3.1 8B GGUF (Best)",
        speed="slow",
        quality="excellent (95-100/100)",
        vram_required="6 GB",
        description="Best quality but slower"
    ),
}

RECIPE_MODELS = {}
CURRENT_MODEL_TYPE = RecipeModelType.LLAMA_1B

print("✓ Model configuration loaded")
print(f"  Default: {MODEL_INFO[CURRENT_MODEL_TYPE].display_name}")

✓ Model configuration loaded
  Default: Llama 3.2 1B (Recommended)


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GPT2LMHeadModel, GPT2Tokenizer, BitsAndBytesConfig
from peft import PeftModel

def load_recipe_model(model_type: RecipeModelType) -> Dict[str, Any]:
    """Load specified recipe generation model"""
    print(f"\n{'🔄'*30}")
    print(f"🔧 load_recipe_model called with: {model_type}")
    print(f"   Model type value: {model_type.value}")
    print(f"   Display name: {MODEL_INFO[model_type].display_name}")
    
    if model_type in RECIPE_MODELS:
        print(f"✅ Using cached {MODEL_INFO[model_type].display_name}")
        cached_model = RECIPE_MODELS[model_type]
        print(f"   Cached model type field: {cached_model.get('type')}")
        print(f"{'🔄'*30}\n")
        return cached_model

    print(f"📥 Loading {MODEL_INFO[model_type].display_name}...")

    if model_type == RecipeModelType.GPT2:
        print("   ➡️  Calling load_gpt2_model()")
        model_dict = load_gpt2_model()
    elif model_type == RecipeModelType.LLAMA_1B:
        print("   ➡️  Calling load_llama_1b_model() 【你的訓練模型】")
        model_dict = load_llama_1b_model()
    elif model_type == RecipeModelType.LLAMA_8B_GGUF:
        print("   ➡️  Calling load_llama_8b_gguf_model()")
        model_dict = load_llama_8b_gguf_model()

    print(f"✅ Model loaded, type field: {model_dict.get('type')}")
    print(f"{'🔄'*30}\n")
    
    RECIPE_MODELS[model_type] = model_dict
    return model_dict


def load_gpt2_model() -> Dict[str, Any]:
    """Load fine-tuned GPT-2 model"""
    print("\n  🤖 Loading GPT-2 model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    finetuned_dir = MODEL_DIR / "finetuned"
    
    print(f"  📂 Checking path: {finetuned_dir}")
    
    if finetuned_dir.exists():
        print("  ✅ Fine-tuned GPT-2 directory found!")
        print("  📥 Loading fine-tuned GPT-2...")
        tokenizer = GPT2Tokenizer.from_pretrained(str(finetuned_dir))
        tokenizer.pad_token = tokenizer.eos_token
        model = GPT2LMHeadModel.from_pretrained(str(finetuned_dir))
        model.to(device)
        model.eval()
        print(f"  ✅ Fine-tuned GPT-2 loaded to {device}")
    else:
        print("  ⚠️  Fine-tuned model not found, using base GPT-2...")
        tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token
        model = GPT2LMHeadModel.from_pretrained("gpt2")
        model.to(device)
        model.eval()
        print(f"  ✅ Base GPT-2 loaded to {device}")
    
    print(f"  🏷️  Returning model_dict with type='gpt2'")
    return {"model": model, "tokenizer": tokenizer, "type": "gpt2", "device": device}


def load_llama_1b_model() -> Dict[str, Any]:
    """Load Llama 3.2 1B fine-tuned model (你訓練的模型) - 使用訓練時的 4-bit 量化"""
    print("\n  🦙 Loading Llama 3.2 1B model with 4-bit quantization...")
    base_model_name = "meta-llama/Llama-3.2-1B-Instruct"
    adapter_path = MODEL_DIR / "llama3_1b_finetuned"
    
    print(f"  📂 Base model: {base_model_name}")
    print(f"  📂 Adapter path: {adapter_path}")
    print(f"  📂 Adapter exists: {adapter_path.exists()}")

    # 使用與訓練時相同的 4-bit 量化配置
    print("  ⚙️  Configuring 4-bit quantization (same as training)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    print("  📥 Loading base model with 4-bit quantization...")
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,  # 使用 4-bit 量化
        device_map="auto",
        trust_remote_code=True
    )
    print(f"  ✅ Base model loaded (4-bit), device: {model.device}")
    print(f"  💾 Memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

    if adapter_path.exists():
        print("  ✅ LoRA adapter found!")
        print("  📥 Loading LoRA adapter...")
        model = PeftModel.from_pretrained(model, str(adapter_path))
        print("  🔧 Merging weights (may have rounding errors due to 4-bit)...")
        model = model.merge_and_unload()
        print("  ✅ Adapter merged - 使用你的訓練模型！")
        print("  ⚠️  Note: Using same 4-bit quantization as training for consistency")
    else:
        print("  ⚠️  Adapter not found - using base model")

    print(f"  🏷️  Returning model_dict with type='llama'")
    print("  ✅ Llama 3.2 1B loaded successfully")
    return {"model": model, "tokenizer": tokenizer, "type": "llama"}


def load_llama_8b_gguf_model() -> Dict[str, Any]:
    """Load Llama 3.1 8B GGUF model"""
    print("\n  🦙 Loading Llama 3.1 8B GGUF model...")
    try:
        from llama_cpp import Llama
    except ImportError:
        raise ImportError("Install llama-cpp-python: pip install llama-cpp-python")

    model_path = MODEL_DIR / "Meta-Llama-3.1-8B-Instruct-Q5_K_M.gguf"
    print(f"  📂 Model path: {model_path}")
    
    llm = Llama(
        model_path=str(model_path),
        n_gpu_layers=20,
        n_ctx=2048,
        n_batch=512,
        verbose=False
    )

    print(f"  🏷️  Returning model_dict with type='gguf'")
    print("  ✅ Llama 3.1 8B GGUF loaded")
    return {"model": llm, "tokenizer": None, "type": "gguf"}

print("✓ Model loading functions defined (with 4-bit quantization for Llama 1B)")

✓ Model loading functions defined (with 4-bit quantization for Llama 1B)


# GPT-2 Recipe Generation Functions

In [6]:
def parse_gpt2_recipe_output(text: str, ingredient: str, cuisine: str, difficulty: str) -> Dict[str, Any]:
    """Parse GPT-2 generated recipe"""
    recipe = {
        'ingredient': ingredient,
        'recipe_title': f'{cuisine} Style {ingredient.title()}',
        'cuisine': cuisine.lower(),
        'difficulty': difficulty.lower(),
        'cooking_time_minutes': 30,
        'servings': 4,
        'ingredients': [ingredient],
        'instructions': ["Prepare and cook the ingredient.", "Serve hot."]
    }
    
    # Extract title
    title_match = re.search(r'<TITLE>\s*(.+?)(?:\n|<)', text)
    if title_match:
        recipe['recipe_title'] = title_match.group(1).strip()
    
    # Extract ingredients
    ing_match = re.search(r'<INGREDIENTS>\s*(.+?)(?:<INSTRUCTIONS>|<|$)', text, re.DOTALL)
    if ing_match:
        ing_text = ing_match.group(1).strip()
        if ';' in ing_text:
            recipe['ingredients'] = [i.strip() for i in ing_text.split(';') if i.strip()]
    
    # Extract instructions
    inst_match = re.search(r'<INSTRUCTIONS>\s*(.+?)(?:<|$)', text, re.DOTALL)
    if inst_match:
        inst_text = inst_match.group(1).strip()
        steps = re.split(r'\d+\.\s*', inst_text)
        recipe['instructions'] = [s.strip() for s in steps if s.strip()]
    
    return recipe


def generate_recipe_gpt2(model_dict: Dict, ingredient: str, cuisine: str, difficulty: str) -> Dict[str, Any]:
    """Generate recipe using GPT-2 (你訓練的模型)"""
    print(f"\n🔍 DEBUG generate_recipe_gpt2 called:")
    print(f"  - model_dict type: {model_dict.get('type')}")
    print(f"  - model class: {type(model_dict['model']).__name__}")
    print(f"  - ingredient: {ingredient}")
    print(f"  - cuisine: {cuisine}")
    
    model = model_dict["model"]
    tokenizer = model_dict["tokenizer"]
    device = model_dict["device"]
    
    # GPT-2 使用特殊的格式標記
    prompt = f"""<INGREDIENT> {ingredient}
<CUISINE> {cuisine}
<TITLE> """
    
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=350,
            temperature=0.5,
            top_k=50,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=2
        )
    
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"  - Generated text length: {len(generated_text)} chars")
    print(f"  - Text preview: {generated_text[:100]}...")
    
    return parse_gpt2_recipe_output(generated_text, ingredient, cuisine, difficulty)

print("✓ GPT-2 recipe generation functions defined")

✓ GPT-2 recipe generation functions defined



# Llama Recipe Generation Functions 


In [7]:


def generate_recipe_from_ingredients(model, tokenizer, ingredients_str, cuisine=None):
    """
    從食材生成食譜（使用目前的微調 Llama 模型）
    這是你訓練時使用的生成函數
    """
    cuisine_hint = f" ({cuisine} style)" if cuisine else ""
    
    # Llama 3.2 使用特殊的對話格式
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant. Generate creative, detailed recipes from given ingredients.<|eot_id|><|start_header_id|>user<|end_header_id|>

Create a delicious recipe using these ingredients: {ingredients_str}{cuisine_hint}.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.8,  # 提高創意
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.1  # 避免重複
    )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    response = full_output.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    response = response.split("<|eot_id|>")[0].strip()
    
    return response


def parse_llama_recipe_output(response: str, ingredient: str, cuisine: str, difficulty: str) -> Dict[str, Any]:
    """
    Format Llama output - 不重新解析，保留原始 Markdown 格式
    只提取標題，其餘內容保持原樣
    """
    lines = response.strip().split('\n')
    
    # 只提取標題 (第一個 # heading)
    title = "Recipe"
    for line in lines:
        if line.startswith('#') and not line.startswith('##'):
            title = line.strip('# ').strip()
            break
    
    # 保留完整的 Markdown 內容
    return {
        "ingredient": ingredient,
        "recipe_title": title,
        "cuisine": cuisine,
        "difficulty": difficulty,
        "cooking_time_minutes": 30,
        "servings": 4,
        "raw_markdown": response,  # 完整的 Markdown 內容
        "ingredients": [ingredient],  # 簡化：只放主要食材
        "instructions": [response]     # 完整內容作為單一項目
    }


def generate_recipe_llama(model_dict: Dict, ingredient: str, cuisine: str, difficulty: str) -> Dict[str, Any]:
    """Generate recipe using Llama model (使用你的訓練模型)"""
    print(f"\n🔍 DEBUG generate_recipe_llama called:")
    print(f"  - model_dict type: {model_dict.get('type')}")
    print(f"  - ingredient: {ingredient}")
    print(f"  - cuisine: {cuisine}")
    
    model = model_dict["model"]
    tokenizer = model_dict["tokenizer"]
    
    print(f"  - model class: {type(model).__name__}")
    print(f"  - model device: {model.device}")
    
    # 使用你提供的新生成函數
    response = generate_recipe_from_ingredients(model, tokenizer, ingredient, cuisine)
    
    print(f"  - Generated response length: {len(response)} chars")
    print(f"  - Response preview: {response[:100]}...")
    
    return parse_llama_recipe_output(response, ingredient, cuisine, difficulty)


def generate_recipe_gguf(model_dict: Dict, ingredient: str, cuisine: str, difficulty: str) -> Dict[str, Any]:
    """Generate recipe using GGUF model (Llama 3.1 8B)"""
    print(f"\n🔍 DEBUG generate_recipe_gguf called:")
    print(f"  - ingredient: {ingredient}")
    
    llm = model_dict["model"]
    cuisine_hint = f" ({cuisine} style)" if cuisine and cuisine != "any" else ""
    
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Create a {difficulty} recipe using: {ingredient}{cuisine_hint}.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# """
    
    output = llm(
        prompt,
        max_tokens=512,
        temperature=0.7,
        top_p=0.9,
        stop=["<|eot_id|>"]
    )
    
    response = output['choices'][0]['text']
    return parse_llama_recipe_output(response, ingredient, cuisine, difficulty)

print("✓ Llama recipe generation functions defined (保留原始 Markdown 格式)")

✓ Llama recipe generation functions defined (保留原始 Markdown 格式)


In [8]:
# ============================================================================
# Model Router - Routes to correct generation function
# ============================================================================

def generate_recipe_with_selected_model(
    ingredient: str,
    cuisine: str = "any",
    difficulty: str = "medium",
    model_type: RecipeModelType = None
) -> Dict[str, Any]:
    """
    Generate recipe using selected model
    
    This is the main entry point for recipe generation.
    It routes to the appropriate generation function based on model type.
    """
    if model_type is None:
        model_type = CURRENT_MODEL_TYPE
    
    print(f"\n{'='*60}")
    print(f"🎯 generate_recipe_with_selected_model called:")
    print(f"  - model_type parameter: {model_type}")
    print(f"  - model_type value: {model_type.value}")
    print(f"  - ingredient: {ingredient}")
    print(f"  - cuisine: {cuisine}")
    print(f"  - difficulty: {difficulty}")
    print(f"{'='*60}")
    
    # Load model (or use cached)
    model_dict = load_recipe_model(model_type)
    
    print(f"\n📦 Model dict returned:")
    print(f"  - type field: {model_dict.get('type')}")
    print(f"  - model class: {type(model_dict.get('model')).__name__}")
    
    # Route to appropriate generation function
    if model_dict["type"] == "gpt2":
        print(f"  ➡️  Routing to generate_recipe_gpt2")
        return generate_recipe_gpt2(model_dict, ingredient, cuisine, difficulty)
    
    elif model_dict["type"] == "llama":
        print(f"  ➡️  Routing to generate_recipe_llama (你的訓練模型)")
        return generate_recipe_llama(model_dict, ingredient, cuisine, difficulty)
    
    elif model_dict["type"] == "gguf":
        print(f"  ➡️  Routing to generate_recipe_gguf")
        return generate_recipe_gguf(model_dict, ingredient, cuisine, difficulty)
    
    else:
        raise ValueError(f"Unknown model type: {model_dict.get('type')}")

print("✓ Model router defined")

✓ Model router defined


In [9]:
# Interactive Model Selector
import ipywidgets as widgets
from IPython.display import display

model_selector = widgets.Dropdown(
    options=[
        ('GPT-2 (Fast)', RecipeModelType.GPT2),
        ('Llama 3.2 1B (Recommended)', RecipeModelType.LLAMA_1B),
        ('Llama 3.1 8B GGUF (Best Quality)', RecipeModelType.LLAMA_8B_GGUF)
    ],
    value=RecipeModelType.LLAMA_1B,
    description='Model:'
)

info_button = widgets.Button(description='Show Info', button_style='info')
info_output = widgets.Output()

def show_model_info(b):
    with info_output:
        info_output.clear_output()
        print("Available Models:\n")
        for model_type, config in MODEL_INFO.items():
            print(f"{config.display_name}")
            print(f"  Speed: {config.speed} | Quality: {config.quality}")
            print(f"  VRAM: {config.vram_required}\n")

info_button.on_click(show_model_info)

def update_current_model(change):
    global CURRENT_MODEL_TYPE
    CURRENT_MODEL_TYPE = change['new']
    print(f"Switched to: {MODEL_INFO[CURRENT_MODEL_TYPE].display_name}")

model_selector.observe(update_current_model, names='value')

display(widgets.VBox([
    widgets.Label('Select Recipe Generation Model:'),
    model_selector,
    info_button,
    info_output
]))

print(f"Current model: {MODEL_INFO[CURRENT_MODEL_TYPE].display_name}")

Current model: Llama 3.2 1B (Recommended)


## 5. Load Nutrition Database

In [10]:
NUTRITION_JSON = DATA_DIR / "nutrition_lookup_full.json"

if NUTRITION_JSON.exists():
    with open(NUTRITION_JSON, 'r', encoding='utf-8') as f:
        NUTRITION_DB = json.load(f)
    print(f"✓ Nutrition database loaded: {len(NUTRITION_DB)} ingredients")
else:
    print(f"⚠ Nutrition database not found, using fallback...")
    NUTRITION_DB = {
        'chicken breast': {'calories': 165, 'protein_g': 31, 'fat_g': 3.6, 'carbs_g': 0},
        'chicken': {'calories': 239, 'protein_g': 27, 'fat_g': 14, 'carbs_g': 0},
        'beef': {'calories': 250, 'protein_g': 26, 'fat_g': 15, 'carbs_g': 0},
        'salmon': {'calories': 208, 'protein_g': 20, 'fat_g': 13, 'carbs_g': 0},
        'tomato': {'calories': 18, 'protein_g': 0.9, 'fat_g': 0.2, 'carbs_g': 3.9},
        'potato': {'calories': 77, 'protein_g': 2, 'fat_g': 0.1, 'carbs_g': 17},
        'egg': {'calories': 155, 'protein_g': 13, 'fat_g': 11, 'carbs_g': 1.1},
        'rice': {'calories': 130, 'protein_g': 2.7, 'fat_g': 0.3, 'carbs_g': 28},
    }
    print(f"✓ Fallback database loaded: {len(NUTRITION_DB)} ingredients")

TYPICAL_WEIGHTS = {
    'chicken breast': 200, 'chicken': 150, 'beef': 200,
    'salmon': 150, 'tomato': 120, 'potato': 180,
    'egg': 50, 'rice': 150
}

✓ Nutrition database loaded: 525 ingredients


## 6. Detection and Helper Functions

In [11]:
def detect_ingredient_clip(image_path: str, confidence_threshold: float = 0.15) -> Dict:
    """Detect ingredient using CLIP (single-ingredient mode)"""
    image = Image.open(image_path).convert('RGB')
    
    inputs = clip_processor(
        text=INGREDIENT_CANDIDATES,
        images=image,
        return_tensors="pt",
        padding=True
    )
    
    with torch.no_grad():
        outputs = clip_model(**inputs)
    
    probs = outputs.logits_per_image.softmax(dim=1)[0]
    top_prob, top_idx = probs.max(0)
    
    if top_prob.item() < confidence_threshold:
        return None
    
    img_width, img_height = image.size
    
    return {
        'class': INGREDIENT_CANDIDATES[top_idx],
        'confidence': top_prob.item(),
        'width': img_width * 0.6,
        'height': img_height * 0.6,
        'x': img_width / 2,
        'y': img_height / 2,
        'detection_method': 'CLIP_single'
    }


def detect_multiple_ingredients_clip(image_path: str, 
                                     object_threshold: float = 0.7,
                                     ingredient_threshold: float = 0.15) -> List[Dict]:
    """Detect multiple ingredients using DETR + CLIP"""
    image = Image.open(image_path).convert('RGB')
    
    inputs = detr_processor(images=image, return_tensors="pt")
    outputs = detr_model(**inputs)
    
    target_sizes = torch.tensor([image.size[::-1]])
    results = detr_processor.post_process_object_detection(
        outputs, 
        target_sizes=target_sizes, 
        threshold=object_threshold
    )[0]
    
    detected_ingredients = []
    
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box_coords = [int(i) for i in box.tolist()]
        x1, y1, x2, y2 = box_coords
        
        cropped = image.crop((x1, y1, x2, y2))
        
        clip_inputs = clip_processor(
            text=INGREDIENT_CANDIDATES,
            images=cropped,
            return_tensors="pt",
            padding=True
        )
        
        with torch.no_grad():
            clip_outputs = clip_model(**clip_inputs)
        
        probs = clip_outputs.logits_per_image.softmax(dim=1)[0]
        top_prob, top_idx = probs.max(0)
        
        if top_prob.item() >= ingredient_threshold:
            detected_ingredients.append({
                'class': INGREDIENT_CANDIDATES[top_idx],
                'confidence': top_prob.item(),
                'width': x2 - x1,
                'height': y2 - y1,
                'x': (x1 + x2) / 2,
                'y': (y1 + y2) / 2,
                'bbox': box_coords,
                'detection_method': 'DETR+CLIP_multi'
            })
    
    return detected_ingredients


def generate_diverse_prompts(ingredient: str, num_recipes: int = 5) -> List[Dict]:
    """Generate diverse cuisine prompts"""
    configurations = [
        {'cuisine': 'Asian', 'difficulty': 'beginner'},
        {'cuisine': 'Western', 'difficulty': 'beginner'},
        {'cuisine': 'Fusion', 'difficulty': 'intermediate'},
        {'cuisine': 'Mediterranean', 'difficulty': 'beginner'},
        {'cuisine': 'any', 'difficulty': 'intermediate'},
    ]
    
    prompts = []
    for i in range(min(num_recipes, len(configurations))):
        config = configurations[i]
        prompts.append({
            'ingredient': ingredient,
            'cuisine': config['cuisine'],
            'difficulty': config['difficulty'],
            'recipe_index': i + 1
        })
    
    return prompts


def estimate_nutrition(ingredient: str, bbox_width: int, bbox_height: int,
                      image_width: int = 640, image_height: int = 640) -> Dict:
    """Estimate nutrition from bounding box"""
    ingredient_lower = ingredient.lower()
    
    typical_weight = TYPICAL_WEIGHTS.get(ingredient_lower, 150)
    bbox_area = bbox_width * bbox_height
    image_area = image_width * image_height
    area_ratio = bbox_area / image_area
    size_multiplier = (area_ratio / 0.25) ** 0.7
    estimated_weight = typical_weight * size_multiplier
    
    if any(m in ingredient_lower for m in ['chicken', 'beef', 'pork', 'salmon', 'fish']):
        serving_size = 120
    elif any(v in ingredient_lower for v in ['potato', 'tomato', 'vegetable']):
        serving_size = 100
    else:
        serving_size = 100
    
    servings = max(1, round(estimated_weight / serving_size * 2) / 2)
    g_per_serving = estimated_weight / servings
    
    nutrition_base = None
    if ingredient in NUTRITION_DB:
        nutrition_base = NUTRITION_DB[ingredient]
    else:
        for key in NUTRITION_DB.keys():
            if key.lower() in ingredient_lower or ingredient_lower in key.lower():
                nutrition_base = NUTRITION_DB[key]
                break
    
    if not nutrition_base:
        return {'success': False, 'error': f'No nutrition data for {ingredient}'}
    
    multiplier = g_per_serving / 100
    calories = nutrition_base['calories'] * multiplier
    
    return {
        'success': True,
        'weight_g': round(estimated_weight, 1),
        'servings': int(servings) if servings.is_integer() else servings,
        'per_serving': {
            'weight_g': round(g_per_serving, 1),
            'calories': round(calories, 0),
            'calories_range': f"{round(calories*0.8, 0):.0f}-{round(calories*1.2, 0):.0f} kcal",
            'protein_g': round(nutrition_base['protein_g'] * multiplier, 1),
            'fat_g': round(nutrition_base['fat_g'] * multiplier, 1),
            'carbs_g': round(nutrition_base['carbs_g'] * multiplier, 1)
        }
    }

print("✓ Helper functions defined")

✓ Helper functions defined


## 7. Main Pipeline Function

In [12]:
def process_ingredient_photo(image_path: str, 
                            confidence_threshold: Optional[float] = None,
                            detection_mode: Optional[str] = None,
                            verbose: bool = True) -> Dict:
    """Complete pipeline: Photo → Recipes with Nutrition"""
    start_time = time.time()
    
    conf_threshold = confidence_threshold if confidence_threshold is not None else INGREDIENT_CONFIDENCE_THRESHOLD
    mode = detection_mode if detection_mode is not None else DETECTION_MODE
    
    if verbose:
        print("\n" + "="*80)
        print(f"RECIPE GENERATION PIPELINE ({MODEL_INFO[CURRENT_MODEL_TYPE].display_name})")
        print("="*80)
    
    # Step 1: Detect ingredients
    try:
        if mode == "single":
            primary = detect_ingredient_clip(image_path, conf_threshold)
            if not primary:
                return {'success': False, 'error': 'No ingredient detected'}
            detected = [primary]
        else:
            detected = detect_multiple_ingredients_clip(
                image_path,
                object_threshold=OBJECT_DETECTION_THRESHOLD,
                ingredient_threshold=conf_threshold
            )
            if not detected:
                return {'success': False, 'error': 'No ingredients detected'}
        
        if verbose:
            print(f"✓ Found {len(detected)} ingredient(s)")
    except Exception as e:
        return {'success': False, 'error': f'Detection failed: {e}'}
    
    # Step 2: Consolidate ingredients
    ingredient_map = {}
    for detection in detected:
        ing_name = detection['class']
        if ing_name not in ingredient_map:
            ingredient_map[ing_name] = {
                'name': ing_name,
                'confidence': detection['confidence'],
                'count': 1,
                'total_area': detection['width'] * detection['height']
            }
        else:
            ingredient_map[ing_name]['confidence'] = max(
                ingredient_map[ing_name]['confidence'],
                detection['confidence']
            )
            ingredient_map[ing_name]['count'] += 1
            ingredient_map[ing_name]['total_area'] += detection['width'] * detection['height']
    
    unique_ingredients = list(ingredient_map.keys())
    combined_ingredient = " and ".join(unique_ingredients)
    
    # Step 3: Generate recipes
    if verbose:
        print(f"Generating {NUM_RECIPES} recipes...")
    
    prompts = generate_diverse_prompts(combined_ingredient, NUM_RECIPES)
    recipes = []
    
    for prompt in prompts:
        recipe = generate_recipe_with_selected_model(
            ingredient=combined_ingredient,
            cuisine=prompt['cuisine'],
            difficulty=prompt['difficulty']
        )
        recipes.append(recipe)
    
    # Step 4: Calculate nutrition
    primary_ing = max(ingredient_map.items(), key=lambda x: x[1]['total_area'])
    primary_name = primary_ing[0]
    primary_area = primary_ing[1]['total_area']
    
    estimated_width = int(primary_area ** 0.5)
    estimated_height = int(primary_area ** 0.5)
    
    nutrition = estimate_nutrition(primary_name, estimated_width, estimated_height)
    
    for recipe in recipes:
        recipe['nutrition'] = nutrition.get('per_serving', {}) if nutrition['success'] else None
    
    elapsed_time = time.time() - start_time
    
    if verbose:
        print(f"✓ Complete in {elapsed_time:.2f}s")
    
    return {
        'success': True,
        'ingredient': {
            'name': combined_ingredient,
            'unique_ingredients': unique_ingredients,
            'primary_ingredient': primary_name,
            'confidence': ingredient_map[primary_name]['confidence']
        },
        'nutrition': nutrition if nutrition['success'] else None,
        'recipes': recipes,
        'num_recipes': len(recipes),
        'processing_time_seconds': elapsed_time
    }

print("✓ Pipeline function defined")

✓ Pipeline function defined


In [13]:
def save_recipe_to_json(recipe: Dict, recipe_id: int, session_id: str = None) -> str:
    """儲存食譜為 JSON 檔案"""
    from datetime import datetime
    
    if session_id is None:
        session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    output_dir = RESULTS_DIR / "recipes_json" / session_id
    output_dir.mkdir(parents=True, exist_ok=True)
    
    filename = f"recipe_{recipe_id:02d}.json"
    filepath = output_dir / filename
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(recipe, f, ensure_ascii=False, indent=2)
    
    print(f"  💾 Saved: {filepath}")
    return str(filepath)


def gradio_detect_ingredients(image, confidence_threshold):
    """Stage 1: Detect ingredients and calculate nutrition"""
    if image is None:
        return "⚠️ Please upload an image first.", "", "{}"
    
    temp_path = RESULTS_DIR / "temp_upload.jpg"
    image.save(temp_path)
    
    start_time = time.time()
    conf_threshold = confidence_threshold if confidence_threshold is not None else INGREDIENT_CONFIDENCE_THRESHOLD
    mode = DETECTION_MODE
    
    try:
        if mode == "single":
            primary = detect_ingredient_clip(str(temp_path), conf_threshold)
            if not primary:
                return f"❌ No ingredient detected", "", "{}"
            detected = [primary]
        else:
            detected = detect_multiple_ingredients_clip(
                str(temp_path),
                object_threshold=OBJECT_DETECTION_THRESHOLD,
                ingredient_threshold=conf_threshold
            )
            if not detected:
                return "❌ No ingredients detected", "", "{}"
    except Exception as e:
        return f"❌ Detection failed: {e}", "", "{}"
    
    # Consolidate
    ingredient_map = {}
    for detection in detected:
        ing_name = detection['class']
        if ing_name not in ingredient_map:
            ingredient_map[ing_name] = {
                'name': ing_name,
                'confidence': detection['confidence'],
                'count': 1,
                'total_area': detection['width'] * detection['height']
            }
        else:
            ingredient_map[ing_name]['confidence'] = max(
                ingredient_map[ing_name]['confidence'],
                detection['confidence']
            )
            ingredient_map[ing_name]['count'] += 1
            ingredient_map[ing_name]['total_area'] += detection['width'] * detection['height']
    
    unique_ingredients = list(ingredient_map.keys())
    combined_ingredient = " and ".join(unique_ingredients)
    
    # Calculate nutrition
    primary_ing = max(ingredient_map.items(), key=lambda x: x[1]['total_area'])
    primary_name = primary_ing[0]
    primary_area = primary_ing[1]['total_area']
    
    estimated_width = int(primary_area ** 0.5)
    estimated_height = int(primary_area ** 0.5)
    nutrition = estimate_nutrition(primary_name, estimated_width, estimated_height)
    
    elapsed_time = time.time() - start_time
    
    detection_result = f"""# 🔍 Detection Results

**Ingredients**: {combined_ingredient}
**Primary**: {primary_name}
**Confidence**: {ingredient_map[primary_name]['confidence']:.1%}
**Time**: {elapsed_time:.2f}s

✅ Detection complete! Click '🍳 Generate Recipes' to continue.
"""
    
    if nutrition['success']:
        nutrition_result = f"""# 🥗 Nutrition Information

**Ingredient**: {primary_name}
**Weight**: {nutrition['weight_g']}g
**Servings**: {nutrition['servings']}

### Per Serving ({nutrition['per_serving']['weight_g']}g)
- Calories: {nutrition['per_serving']['calories']:.0f} kcal
- Protein: {nutrition['per_serving']['protein_g']}g
- Fat: {nutrition['per_serving']['fat_g']}g
- Carbs: {nutrition['per_serving']['carbs_g']}g
"""
    else:
        nutrition_result = "⚠️ Nutrition data not available"
    
    detection_data = {
        'combined_ingredient': combined_ingredient,
        'unique_ingredients': unique_ingredients,
        'primary_name': primary_name,
        'nutrition': nutrition if nutrition['success'] else None
    }
    
    return detection_result, nutrition_result, json.dumps(detection_data)


def gradio_generate_recipes(detection_data_json, selected_model):
    """Stage 2: Generate recipes with selected model (儲存 JSON + 顯示)"""
    from datetime import datetime
    
    if not detection_data_json or detection_data_json == "{}":
        yield "⚠️ Please detect ingredients first"
        return
    
    try:
        detection_data = json.loads(detection_data_json)
    except:
        yield "❌ Invalid detection data"
        return
    
    combined_ingredient = detection_data['combined_ingredient']
    session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Parse selected model
    model_type = RecipeModelType.LLAMA_1B  # Default
    print(f"\n🔍 DEBUG: Selected model string: '{selected_model}'")
    
    if "GPT-2" in selected_model:
        model_type = RecipeModelType.GPT2
        print(f"✓ Matched GPT-2, model_type = {model_type}")
    elif "Llama 3.2 1B" in selected_model:
        model_type = RecipeModelType.LLAMA_1B
        print(f"✓ Matched Llama 3.2 1B, model_type = {model_type}")
    elif "Llama 3.1 8B" in selected_model:
        model_type = RecipeModelType.LLAMA_8B_GGUF
        print(f"✓ Matched Llama 3.1 8B, model_type = {model_type}")
    
    print(f"🎯 Final model_type: {model_type}")
    print(f"📝 Model enum value: {model_type.value}")
    print(f"📁 Session ID: {session_id}")
    
    yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Selected**: {selected_model}
**Model Type**: {model_type.value}
**Session**: {session_id}

⏳ Starting recipe generation...
"""
    
    prompts = generate_diverse_prompts(combined_ingredient, NUM_RECIPES)
    recipes = []
    
    progress_msg = f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Type**: {model_type.value}
**Session**: {session_id}

"""
    
    for idx, prompt in enumerate(prompts, 1):
        progress_msg += f"⏳ Generating {prompt['cuisine']} recipe ({idx}/{NUM_RECIPES})...\n"
        yield progress_msg
        
        print(f"\n🍳 Generating recipe {idx}/{NUM_RECIPES} with model_type={model_type}")
        
        # 生成食譜
        recipe = generate_recipe_with_selected_model(
            ingredient=combined_ingredient,
            cuisine=prompt['cuisine'],
            difficulty=prompt['difficulty'],
            model_type=model_type
        )
        
        # 加入營養資訊
        if detection_data.get('nutrition'):
            recipe['nutrition'] = detection_data['nutrition'].get('per_serving', {})
        
        # 💾 儲存 JSON
        recipe['session_id'] = session_id
        recipe['model_used'] = selected_model
        recipe['model_type'] = model_type.value
        recipe['generated_at'] = datetime.now().isoformat()
        
        json_path = save_recipe_to_json(recipe, idx, session_id)
        recipe['json_path'] = json_path
        
        print(f"✓ Recipe {idx} generated and saved: {recipe.get('recipe_title', 'N/A')}")
        
        recipes.append(recipe)
        
        progress_msg = progress_msg.replace(
            f"⏳ Generating {prompt['cuisine']} recipe ({idx}/{NUM_RECIPES})...",
            f"✅ Generated {prompt['cuisine']} recipe ({idx}/{NUM_RECIPES})"
        )
        yield progress_msg
    
    # Final results - 從 JSON 讀取並顯示
    final_result = f"""# 🍳 Generated Recipes ({len(recipes)})

**Ingredients**: {combined_ingredient}
**Model Used**: {selected_model}
**Model Type**: {model_type.value}
**Session**: {session_id}
**Saved to**: `{RESULTS_DIR / 'recipes_json' / session_id}`

---

"""
    
    for idx, recipe in enumerate(recipes, 1):
        # 如果有 raw_markdown (Llama 模型)，直接顯示完整內容，不重複標題
        if recipe.get('raw_markdown'):
            raw_md = recipe['raw_markdown']
            
            # 從 raw_markdown 中移除第一個 # 標題行（避免重複）
            lines = raw_md.split('\n')
            content_lines = []
            skip_first_heading = False
            
            for line in lines:
                # 跳過第一個 # 開頭的行（通常是標題）
                if not skip_first_heading and line.startswith('#') and not line.startswith('##'):
                    skip_first_heading = True
                    continue
                content_lines.append(line)
            
            cleaned_markdown = '\n'.join(content_lines)
            
            final_result += f"""## {idx}. {recipe['recipe_title']}

**Cuisine**: {recipe.get('cuisine', 'N/A')} | **Difficulty**: {recipe.get('difficulty', 'N/A')} | **Time**: {recipe.get('cooking_time_minutes', 30)} min

"""
            if recipe.get('nutrition'):
                n = recipe['nutrition']
                final_result += f"**Nutrition**: {n['calories']:.0f} kcal | Protein: {n['protein_g']}g\n\n"
            
            final_result += cleaned_markdown + "\n\n"
        else:
            # GPT-2 模型的舊格式
            final_result += f"""## {idx}. {recipe['recipe_title']}

**Cuisine**: {recipe.get('cuisine', 'N/A')} | **Difficulty**: {recipe.get('difficulty', 'N/A')} | **Time**: {recipe.get('cooking_time_minutes', 30)} min

"""
            if recipe.get('nutrition'):
                n = recipe['nutrition']
                final_result += f"**Nutrition**: {n['calories']:.0f} kcal | Protein: {n['protein_g']}g\n\n"
            
            if recipe.get('ingredients'):
                final_result += "### Ingredients\n"
                for ing in recipe['ingredients'][:8]:
                    final_result += f"- {ing}\n"
                final_result += "\n"
            
            if recipe.get('instructions'):
                final_result += "### Instructions\n"
                for step_idx, step in enumerate(recipe['instructions'][:6], 1):
                    final_result += f"{step_idx}. {step}\n"
                final_result += "\n"
        
        final_result += f"📄 **JSON**: `{recipe.get('json_path', 'N/A')}`\n\n"
        final_result += "---\n\n"
    
    final_result += f"""
✅ All recipes generated and saved!

📁 **JSON Files Location:**
`{RESULTS_DIR / 'recipes_json' / session_id}`

🔍 **Debug Info:**
- Model selected: {selected_model}
- Model type used: {model_type.value}
- Session ID: {session_id}
"""
    yield final_result

print("✓ Gradio functions defined (移除重複標題)")

✓ Gradio functions defined (移除重複標題)


In [14]:
try:
    import gradio as gr
    print("✓ Gradio already installed")
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
    import gradio as gr
    print("✓ Gradio installed")

✓ Gradio already installed


In [15]:
def save_recipe_to_json(recipe: Dict, recipe_id: int, session_id: str = None) -> str:
    """儲存食譜為 JSON 檔案"""
    from datetime import datetime
    
    if session_id is None:
        session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    output_dir = RESULTS_DIR / "recipes_json" / session_id
    output_dir.mkdir(parents=True, exist_ok=True)
    
    filename = f"recipe_{recipe_id:02d}.json"
    filepath = output_dir / filename
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(recipe, f, ensure_ascii=False, indent=2)
    
    print(f"  💾 Saved: {filepath}")
    return str(filepath)


def gradio_detect_ingredients(image, confidence_threshold):
    """Stage 1: Detect ingredients and calculate nutrition"""
    if image is None:
        return "⚠️ Please upload an image first.", "", "{}"
    
    temp_path = RESULTS_DIR / "temp_upload.jpg"
    image.save(temp_path)
    
    start_time = time.time()
    conf_threshold = confidence_threshold if confidence_threshold is not None else INGREDIENT_CONFIDENCE_THRESHOLD
    mode = DETECTION_MODE
    
    try:
        if mode == "single":
            primary = detect_ingredient_clip(str(temp_path), conf_threshold)
            if not primary:
                return f"❌ No ingredient detected", "", "{}"
            detected = [primary]
        else:
            detected = detect_multiple_ingredients_clip(
                str(temp_path),
                object_threshold=OBJECT_DETECTION_THRESHOLD,
                ingredient_threshold=conf_threshold
            )
            if not detected:
                return "❌ No ingredients detected", "", "{}"
    except Exception as e:
        return f"❌ Detection failed: {e}", "", "{}"
    
    # Consolidate
    ingredient_map = {}
    for detection in detected:
        ing_name = detection['class']
        if ing_name not in ingredient_map:
            ingredient_map[ing_name] = {
                'name': ing_name,
                'confidence': detection['confidence'],
                'count': 1,
                'total_area': detection['width'] * detection['height']
            }
        else:
            ingredient_map[ing_name]['confidence'] = max(
                ingredient_map[ing_name]['confidence'],
                detection['confidence']
            )
            ingredient_map[ing_name]['count'] += 1
            ingredient_map[ing_name]['total_area'] += detection['width'] * detection['height']
    
    unique_ingredients = list(ingredient_map.keys())
    combined_ingredient = " and ".join(unique_ingredients)
    
    # Calculate nutrition
    primary_ing = max(ingredient_map.items(), key=lambda x: x[1]['total_area'])
    primary_name = primary_ing[0]
    primary_area = primary_ing[1]['total_area']
    
    estimated_width = int(primary_area ** 0.5)
    estimated_height = int(primary_area ** 0.5)
    nutrition = estimate_nutrition(primary_name, estimated_width, estimated_height)
    
    elapsed_time = time.time() - start_time
    
    detection_result = f"""# 🔍 Detection Results

**Ingredients**: {combined_ingredient}
**Primary**: {primary_name}
**Confidence**: {ingredient_map[primary_name]['confidence']:.1%}
**Time**: {elapsed_time:.2f}s

✅ Detection complete! Click '🍳 Generate Recipes' to continue.
"""
    
    if nutrition['success']:
        nutrition_result = f"""# 🥗 Nutrition Information

**Ingredient**: {primary_name}
**Weight**: {nutrition['weight_g']}g
**Servings**: {nutrition['servings']}

### Per Serving ({nutrition['per_serving']['weight_g']}g)
- Calories: {nutrition['per_serving']['calories']:.0f} kcal
- Protein: {nutrition['per_serving']['protein_g']}g
- Fat: {nutrition['per_serving']['fat_g']}g
- Carbs: {nutrition['per_serving']['carbs_g']}g
"""
    else:
        nutrition_result = "⚠️ Nutrition data not available"
    
    detection_data = {
        'combined_ingredient': combined_ingredient,
        'unique_ingredients': unique_ingredients,
        'primary_name': primary_name,
        'nutrition': nutrition if nutrition['success'] else None
    }
    
    return detection_result, nutrition_result, json.dumps(detection_data)


def gradio_generate_recipes(detection_data_json, selected_model):
    """Stage 2: Generate recipes with selected model (儲存 JSON + 顯示)"""
    from datetime import datetime
    
    if not detection_data_json or detection_data_json == "{}":
        yield "⚠️ Please detect ingredients first"
        return
    
    try:
        detection_data = json.loads(detection_data_json)
    except:
        yield "❌ Invalid detection data"
        return
    
    combined_ingredient = detection_data['combined_ingredient']
    session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Parse selected model
    model_type = RecipeModelType.LLAMA_1B  # Default
    print(f"\n🔍 DEBUG: Selected model string: '{selected_model}'")
    
    if "GPT-2" in selected_model:
        model_type = RecipeModelType.GPT2
        print(f"✓ Matched GPT-2, model_type = {model_type}")
    elif "Llama 3.2 1B" in selected_model:
        model_type = RecipeModelType.LLAMA_1B
        print(f"✓ Matched Llama 3.2 1B, model_type = {model_type}")
    elif "Llama 3.1 8B" in selected_model:
        model_type = RecipeModelType.LLAMA_8B_GGUF
        print(f"✓ Matched Llama 3.1 8B, model_type = {model_type}")
    
    print(f"🎯 Final model_type: {model_type}")
    print(f"📝 Model enum value: {model_type.value}")
    print(f"📁 Session ID: {session_id}")
    
    yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Selected**: {selected_model}
**Model Type**: {model_type.value}
**Session**: {session_id}

⏳ Starting recipe generation...
"""
    
    prompts = generate_diverse_prompts(combined_ingredient, NUM_RECIPES)
    recipes = []
    
    progress_msg = f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Type**: {model_type.value}
**Session**: {session_id}

"""
    
    for idx, prompt in enumerate(prompts, 1):
        progress_msg += f"⏳ Generating {prompt['cuisine']} recipe ({idx}/{NUM_RECIPES})...\n"
        yield progress_msg
        
        print(f"\n🍳 Generating recipe {idx}/{NUM_RECIPES} with model_type={model_type}")
        
        # 生成食譜
        recipe = generate_recipe_with_selected_model(
            ingredient=combined_ingredient,
            cuisine=prompt['cuisine'],
            difficulty=prompt['difficulty'],
            model_type=model_type
        )
        
        # 加入營養資訊
        if detection_data.get('nutrition'):
            recipe['nutrition'] = detection_data['nutrition'].get('per_serving', {})
        
        # 💾 儲存 JSON
        recipe['session_id'] = session_id
        recipe['model_used'] = selected_model
        recipe['model_type'] = model_type.value
        recipe['generated_at'] = datetime.now().isoformat()
        
        json_path = save_recipe_to_json(recipe, idx, session_id)
        recipe['json_path'] = json_path
        
        print(f"✓ Recipe {idx} generated and saved: {recipe.get('recipe_title', 'N/A')}")
        
        recipes.append(recipe)
        
        progress_msg = progress_msg.replace(
            f"⏳ Generating {prompt['cuisine']} recipe ({idx}/{NUM_RECIPES})...",
            f"✅ Generated {prompt['cuisine']} recipe ({idx}/{NUM_RECIPES})"
        )
        yield progress_msg
    
    # Final results - 從 JSON 讀取並顯示
    final_result = f"""# 🍳 Generated Recipes ({len(recipes)})

**Ingredients**: {combined_ingredient}
**Model Used**: {selected_model}
**Model Type**: {model_type.value}
**Session**: {session_id}
**Saved to**: `{RESULTS_DIR / 'recipes_json' / session_id}`

---

"""
    
    for idx, recipe in enumerate(recipes, 1):
        # 如果有 raw_markdown (Llama 模型)，直接顯示完整內容，不重複標題
        if recipe.get('raw_markdown'):
            raw_md = recipe['raw_markdown']
            
            # 從 raw_markdown 中移除第一個 # 標題行（避免重複）
            lines = raw_md.split('\n')
            content_lines = []
            skip_first_heading = False
            
            for line in lines:
                # 跳過第一個 # 開頭的行（通常是標題）
                if not skip_first_heading and line.startswith('#') and not line.startswith('##'):
                    skip_first_heading = True
                    continue
                content_lines.append(line)
            
            cleaned_markdown = '\n'.join(content_lines)
            
            final_result += f"""## {idx}. {recipe['recipe_title']}

**Cuisine**: {recipe.get('cuisine', 'N/A')} | **Difficulty**: {recipe.get('difficulty', 'N/A')} | **Time**: {recipe.get('cooking_time_minutes', 30)} min

"""
            if recipe.get('nutrition'):
                n = recipe['nutrition']
                final_result += f"**Nutrition**: {n['calories']:.0f} kcal | Protein: {n['protein_g']}g\n\n"
            
            final_result += cleaned_markdown + "\n\n"
        else:
            # GPT-2 模型的舊格式
            final_result += f"""## {idx}. {recipe['recipe_title']}

**Cuisine**: {recipe.get('cuisine', 'N/A')} | **Difficulty**: {recipe.get('difficulty', 'N/A')} | **Time**: {recipe.get('cooking_time_minutes', 30)} min

"""
            if recipe.get('nutrition'):
                n = recipe['nutrition']
                final_result += f"**Nutrition**: {n['calories']:.0f} kcal | Protein: {n['protein_g']}g\n\n"
            
            if recipe.get('ingredients'):
                final_result += "### Ingredients\n"
                for ing in recipe['ingredients'][:8]:
                    final_result += f"- {ing}\n"
                final_result += "\n"
            
            if recipe.get('instructions'):
                final_result += "### Instructions\n"
                for step_idx, step in enumerate(recipe['instructions'][:6], 1):
                    final_result += f"{step_idx}. {step}\n"
                final_result += "\n"
        
        final_result += f"📄 **JSON**: `{recipe.get('json_path', 'N/A')}`\n\n"
        final_result += "---\n\n"
    
    final_result += f"""
✅ All recipes generated and saved!

📁 **JSON Files Location:**
`{RESULTS_DIR / 'recipes_json' / session_id}`

🔍 **Debug Info:**
- Model selected: {selected_model}
- Model type used: {model_type.value}
- Session ID: {session_id}
"""
    yield final_result

print("✓ Gradio functions updated (移除重複標題 + JSON 儲存)")

✓ Gradio functions updated (移除重複標題 + JSON 儲存)


In [16]:
# ============================================================================
# Recipe JSON 修复函数 - 在生成后立即修复问题
# ============================================================================

def extract_title_from_markdown(markdown_text: str) -> str:
    """从 Markdown 中提取真正的标题"""
    if not markdown_text:
        return "Untitled Recipe"
    
    lines = markdown_text.split('\n')
    for line in lines:
        # 找第一个 # 开头的行作为标题
        if line.strip().startswith('#') and not line.strip().startswith('##'):
            title = line.strip('#').strip()
            # 如果标题看起来像是配料（包含数字、单位等），跳过
            if re.search(r'^\d+(/\d+)?\s+(c\.|Tbsp\.|tsp\.|lb\.|oz\.)', title):
                continue
            return title
    
    return "Untitled Recipe"


def fix_recipe_json(recipe: Dict) -> Dict:
    """
    修复单个食谱的问题
    - 修复错误的标题（配料名称 → 真正的标题）
    - 修复格式问题
    - 检测截断内容
    """
    fixed = recipe.copy()
    
    # 1. 修复错误的标题（如 "1 1/2 c. all-purpose flour"）
    current_title = fixed.get('recipe_title', '')
    is_ingredient_format = bool(re.search(r'^\d+(/\d+)?\s+(c\.|Tbsp\.|tsp\.|lb\.|oz\.)', current_title))
    
    if is_ingredient_format:
        print(f"  ⚠️  Bad title detected: '{current_title}'")
        
        # 从 raw_markdown 中提取真正的标题
        if fixed.get('raw_markdown'):
            new_title = extract_title_from_markdown(fixed['raw_markdown'])
            fixed['recipe_title'] = new_title
            print(f"  ✅ Fixed → '{new_title}'")
        else:
            # 生成一个通用标题
            ingredient = fixed.get('ingredient', 'Ingredient')
            cuisine = fixed.get('cuisine', 'Style').capitalize()
            fixed['recipe_title'] = f"{cuisine} {ingredient} Recipe"
            print(f"  ✅ Generated → '{fixed['recipe_title']}'")
    
    # 2. 修复配料列表格式（# 1 cup flour → - 1 cup flour）
    if fixed.get('raw_markdown'):
        raw_md = fixed['raw_markdown']
        if re.search(r'^#\s+\d+', raw_md, re.MULTILINE):
            fixed_md = re.sub(r'^#\s+', '- ', raw_md, flags=re.MULTILINE)
            fixed['raw_markdown'] = fixed_md
            print(f"  ✅ Fixed ingredients format")
    
    # 3. 检查内容是否被截断
    if fixed.get('raw_markdown'):
        raw_md = fixed['raw_markdown'].strip()
        if raw_md.endswith(('Use', 'Add', 'Pour', 'Mix', 'Stir')):
            fixed['_warning'] = 'Content may be truncated'
            print(f"  ⚠️  Content appears truncated")
    
    return fixed


def gradio_generate_recipes_with_fix(detection_data_json, selected_model):
    """
    改进版生成函数：生成 → 保存 → 读取 → 修复 → 重新保存 → 显示
    """
    from datetime import datetime
    
    if not detection_data_json or detection_data_json == "{}":
        yield "⚠️ Please detect ingredients first"
        return
    
    try:
        detection_data = json.loads(detection_data_json)
    except:
        yield "❌ Invalid detection data"
        return
    
    combined_ingredient = detection_data['combined_ingredient']
    session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Parse selected model
    model_type = RecipeModelType.LLAMA_1B
    if "GPT-2" in selected_model:
        model_type = RecipeModelType.GPT2
    elif "Llama 3.2 1B" in selected_model:
        model_type = RecipeModelType.LLAMA_1B
    elif "Llama 3.1 8B" in selected_model:
        model_type = RecipeModelType.LLAMA_8B_GGUF
    
    print(f"🎯 Model: {model_type.value} | Session: {session_id}")
    
    yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Session**: {session_id}

⏳ Step 1/3: Generating recipes...
"""
    
    prompts = generate_diverse_prompts(combined_ingredient, NUM_RECIPES)
    recipes = []
    
    # ========== 步骤 1: 生成 & 保存 ==========
    for idx, prompt in enumerate(prompts, 1):
        recipe = generate_recipe_with_selected_model(
            ingredient=combined_ingredient,
            cuisine=prompt['cuisine'],
            difficulty=prompt['difficulty'],
            model_type=model_type
        )
        
        if detection_data.get('nutrition'):
            recipe['nutrition'] = detection_data['nutrition'].get('per_serving', {})
        
        recipe['session_id'] = session_id
        recipe['model_used'] = selected_model
        recipe['model_type'] = model_type.value
        recipe['generated_at'] = datetime.now().isoformat()
        
        json_path = save_recipe_to_json(recipe, idx, session_id)
        recipe['json_path'] = json_path
        recipes.append(recipe)
        
        yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Session**: {session_id}

⏳ Step 1/3: Generated {idx}/{NUM_RECIPES} recipes...
"""
    
    # ========== 步骤 2: 读取 & 修复 & 重新保存 ==========
    yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Session**: {session_id}

✅ Step 1/3: Complete
⏳ Step 2/3: Fixing recipes...
"""
    
    session_dir = RESULTS_DIR / "recipes_json" / session_id
    fixed_recipes = []
    
    for idx in range(1, len(recipes) + 1):
        json_file = session_dir / f"recipe_{idx:02d}.json"
        
        with open(json_file, 'r', encoding='utf-8') as f:
            recipe = json.load(f)
        
        print(f"\n🔧 Fixing recipe {idx}: {recipe.get('recipe_title', 'Unknown')}")
        fixed_recipe = fix_recipe_json(recipe)
        
        with open(json_file, 'w', encoding='utf-8') as f:
            json.dump(fixed_recipe, f, ensure_ascii=False, indent=2)
        
        fixed_recipes.append(fixed_recipe)
    
    # ========== 步骤 3: 显示 ==========
    yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Session**: {session_id}

✅ Step 1/3: Generated {len(recipes)} recipes
✅ Step 2/3: Fixed all recipes
⏳ Step 3/3: Displaying results...
"""
    
    final_result = f"""# 🍳 Generated & Fixed Recipes ({len(fixed_recipes)})

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}
**Session**: {session_id}

---

"""
    
    for idx, recipe in enumerate(fixed_recipes, 1):
        final_result += f"""## {idx}. {recipe['recipe_title']}

**Cuisine**: {recipe.get('cuisine', 'N/A')} | **Difficulty**: {recipe.get('difficulty', 'N/A')} | **Time**: {recipe.get('cooking_time_minutes', 30)} min

"""
        
        if recipe.get('nutrition'):
            n = recipe['nutrition']
            final_result += f"**Nutrition**: {n.get('calories', 0):.0f} kcal | Protein: {n.get('protein_g', 0)}g\n\n"
        
        if recipe.get('raw_markdown'):
            raw_md = recipe['raw_markdown']
            lines = raw_md.split('\n')
            content_lines = []
            skip_first_heading = False
            
            for line in lines:
                if not skip_first_heading and line.startswith('#') and not line.startswith('##'):
                    skip_first_heading = True
                    continue
                content_lines.append(line)
            
            cleaned_markdown = '\n'.join(content_lines)
            final_result += cleaned_markdown + "\n\n"
        
        if recipe.get('_warning'):
            final_result += f"⚠️ **{recipe['_warning']}**\n\n"
        
        final_result += f"📄 `{recipe.get('json_path', 'N/A')}`\n\n---\n\n"
    
    final_result += f"""
✅ All recipes generated, fixed, and saved!

📁 **Location**: `{session_dir}`

🔧 **Fixes Applied**:
- ✅ Corrected wrong titles
- ✅ Fixed ingredient formatting
- ✅ Detected truncated content
"""
    
    yield final_result

print("✓ Recipe fixer integrated! Use gradio_generate_recipes_with_fix() for auto-fixing")

✓ Recipe fixer integrated! Use gradio_generate_recipes_with_fix() for auto-fixing


In [ ]:
# Create Gradio Interface
with gr.Blocks(title="🍳 cAIuldron - AI Recipe Generator", theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    # 🍳 cAIuldron - AI Recipe Generator
    
    Transform ingredient photos into delicious recipes!
    
    **100% Free • Runs Locally • Multi-Ingredient Detection**
    """)
    
    detection_data_state = gr.State(value="{}")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Upload Photo")
            image_input = gr.Image(type="pil", label="Ingredient Photo")
            
            gr.Markdown("### ⚙️ Detection Settings")
            confidence_slider = gr.Slider(
                minimum=0.05,
                maximum=0.50,
                value=0.15,
                step=0.01,
                label="Confidence Threshold"
            )
            
            detect_btn = gr.Button("🔍 Detect Ingredients", variant="primary", size="lg")
            
            gr.Markdown("### 🤖 Recipe Generation Model")
            model_selector = gr.Dropdown(
                choices=[
                    "GPT-2 (Fast) ✅ 你訓練的",
                    "Llama 3.2 1B (Recommended) ✅ 你訓練的",
                    "Llama 3.1 8B GGUF (Best Quality)"
                ],
                value="Llama 3.2 1B (Recommended) ✅ 你訓練的",
                label="選擇模型 / Select Model",
                info="兩個模型都是你訓練好的！"
            )
            
            generate_btn = gr.Button("🍳 Generate Recipes", variant="secondary", size="lg")
            
            gr.Markdown("""
            ### 💡 使用流程:
            1. **檢測** (~1s) - 偵測食材
            2. **選擇模型** - 選擇你訓練的模型
            3. **生成** (~30-60s) - 生成食譜
            
            ### 🤖 模型對比:
            - **GPT-2**: 快速, 你的訓練模型 ✅
            - **Llama 1B**: 推薦, 你的訓練模型 ✅ (最佳品質)
            - **Llama 8B**: 基礎模型, 未訓練 (最慢但品質好)
            """)
        
        with gr.Column(scale=2):
            gr.Markdown("### 📊 Results")
            
            with gr.Tabs():
                with gr.Tab("🔍 Detection"):
                    detection_output = gr.Markdown(value="上傳圖片並點擊 '🔍 Detect Ingredients'")
                
                with gr.Tab("🥗 Nutrition"):
                    nutrition_output = gr.Markdown(value="營養資訊會在偵測後顯示")
                
                with gr.Tab("🍳 Recipes"):
                    recipes_output = gr.Markdown(value="選擇模型並點擊 '🍳 Generate Recipes'")
    
    gr.Markdown("""
    ---
    **Powered by:** 
    - 🔍 CLIP + DETR (Ingredient Detection)
    - 🍴 GPT-2 / Llama 3.2 1B 
    - 🥗 USDA FoodData Central (Nutrition Database)
    """)
    
    # Connect buttons
    detect_btn.click(
        fn=gradio_detect_ingredients,
        inputs=[image_input, confidence_slider],
        outputs=[detection_output, nutrition_output, detection_data_state]
    )
    
    generate_btn.click(
        fn=gradio_generate_recipes_with_fix,
        inputs=[detection_data_state, model_selector],
        outputs=[recipes_output]
    )

print("\n" + "="*80)
print("🚀 LAUNCHING WEB INTERFACE")
print("="*80)
print("\nYour Trained Models:")
print("  ✅ GPT-2 Fine-tuned (Fast)")
print("  ✅ Llama 3.2 1B Fine-tuned with LoRA (Recommended)")
print("  📦 Llama 3.1 8B GGUF (Base model, not trained)")
print("\nFeatures:")
print("  🔍 Multi-ingredient detection (CLIP + DETR)")
print("  🤖 3 model options (2 are your trained models)")
print("  🥗 Nutrition estimation (529 ingredients)")
print("\nThe interface will open at: http://127.0.0.1:7861")
print("\nTo stop: Press Stop button or Kernel → Interrupt")
print("="*80 + "\n")

app.launch(
    inbrowser=True,
    server_port=7861,
    share=True
)


🚀 LAUNCHING WEB INTERFACE

Your Trained Models:
  ✅ GPT-2 Fine-tuned (Fast)
  ✅ Llama 3.2 1B Fine-tuned with LoRA (Recommended)
  📦 Llama 3.1 8B GGUF (Base model, not trained)

Features:
  🔍 Multi-ingredient detection (CLIP + DETR)
  🤖 3 model options (2 are your trained models)
  🥗 Nutrition estimation (529 ingredients)

The interface will open at: http://127.0.0.1:7861

To stop: Press Stop button or Kernel → Interrupt

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://27a5e24219706e508f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🎯 Model: llama3.2-1b | Session: 20251121_185357

🎯 generate_recipe_with_selected_model called:
  - model_type parameter: RecipeModelType.LLAMA_1B
  - model_type value: llama3.2-1b
  - ingredient: Chicken breast
  - cuisine: Asian
  - difficulty: beginner

🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄
🔧 load_recipe_model called with: RecipeModelType.LLAMA_1B
   Model type value: llama3.2-1b
   Display name: Llama 3.2 1B (Recommended)
📥 Loading Llama 3.2 1B (Recommended)...
   ➡️  Calling load_llama_1b_model() 【你的訓練模型】

  🦙 Loading Llama 3.2 1B model with 4-bit quantization...
  📂 Base model: meta-llama/Llama-3.2-1B-Instruct
  📂 Adapter path: c:\Users\Champion\Documents\GitHub\cAIuldron\models\recipe_generation\llama3_1b_finetuned
  📂 Adapter exists: True
  ⚙️  Configuring 4-bit quantization (same as training)...
  📥 Loading base model with 4-bit quantization...
  ✅ Base model loaded (4-bit), device: cuda:0
  💾 Memory footprint: 0.94 GB
  ✅ LoRA adapter found!
  📥 Loading LoRA adapter...
  🔧 Merging we